In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

"""
============================= 조정할 수 있는 옵션 =============================
 * seg_len : 세그먼트 길이. 학습에 사용될 영상의 기준 길이를 지정. 단위는 프레임.
 * hidden_dim : hidden layer의 차원
 * num_epochs : 에포크 수
 * test_size : 테스트에 사용될 모델의 수
 * 반복 훈련 번호 지정 위치에서 range() 값으로 훈련에 쓰일 csv 파일 범위 설정 가능.
"""

'\n============================= 조정할 수 있는 옵션 =============================\n * seg_len : 세그먼트 길이. 학습에 사용될 영상의 기준 길이를 지정. 단위는 프레임.\n * hidden_dim : hidden layer의 차원\n * num_epochs : 에포크 수\n * test_size : 테스트에 사용될 모델의 수\n * 반복 훈련 번호 지정 위치에서 range() 값으로 훈련에 쓰일 csv 파일 범위 설정 가능.\n'

In [2]:
base_dir = "./"
categories = ["Normal", "Warning", "Fall"]
labels_map = {"Normal": 0, "Warning": 1, "Fall": 2}

def load_data(base_dir, categories, seg_len = 90):

    data = []
    labels = []
    label_map = {category: idx for idx, category in enumerate(categories)}

    
    category_path = os.path.join(base_dir, "coordinate") # 하위 경로 생성

    print(f"checking files in: {category_path}")

    for i in range(1, 400):                                            # 반복 훈련 번호 지정
        file_path = os.path.join(category_path, f"keypoints_{i}.csv") # csv 접근 경로 생성
        print(i)

        if os.path.exists(file_path):
            keypoints_csv = pd.read_csv(file_path)

            keypoints_csv_normal = keypoints_csv[keypoints_csv["label"] == "Normal"]
            keypoints_csv_warning = keypoints_csv[keypoints_csv["label"] == "Warning"]
            keypoints_csv_fall = keypoints_csv[keypoints_csv["label"] == "Fall"]

            keypoints_csv_normal = keypoints_csv_normal.drop(["frame_id","frame_path", "label"], axis = 1)
            keypoints_csv_warning = keypoints_csv_warning.drop(["frame_id","frame_path", "label"], axis = 1)
            keypoints_csv_fall = keypoints_csv_fall.drop(["frame_id","frame_path", "label"], axis = 1)

            keypoints_normal = keypoints_csv_normal.values.tolist() # csv 파일의 vlaue들을 list 형태로 변환
            keypoints_warning = keypoints_csv_warning.values.tolist()
            keypoints_fall = keypoints_csv_fall.values.tolist()

            if len(keypoints_normal) > seg_len:
                keypoints_normal = keypoints_normal[:seg_len]

            elif len(keypoints_normal) < seg_len:
                pad_width_normal = seg_len - len(keypoints_normal)
                keypoints_normal = np.pad(keypoints_normal, ((0, pad_width_normal), (0, 0)), mode = "constant")

            if len(keypoints_warning) > seg_len:
                keypoints_warning = keypoints_warning[:seg_len]

            elif len(keypoints_warning) < seg_len:
                pad_width_warning = seg_len - len(keypoints_warning)
                keypoints_warning = np.pad(keypoints_warning, ((0, pad_width_warning), (0, 0)), mode = "constant")

            if len(keypoints_fall) > seg_len:
                keypoints_fall = keypoints_fall[:seg_len]

            elif len(keypoints_fall) < seg_len:
                pad_width_fall = seg_len - len(keypoints_fall)
                keypoints_fall = np.pad(keypoints_fall, ((0, pad_width_fall), (0, 0)), mode = "constant")

            data.append(keypoints_normal)
            data.append(keypoints_warning)
            data.append(keypoints_fall)

            for category in categories:
                labels.append(label_map[category])
        
        else:
            print(f"file not found: {file_path}")

    return np.array(data), np.array(labels)


# data, labels = load_data(base_dir, categories)

X, y = load_data(base_dir, categories)

# print(f"Data shape: {data.shape}, Labels shape: {labels.shape}")

print(f"Data shape: {X.shape}, Labels shape: {y.shape}")


checking files in: ./coordinate
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
file not found: ./coordinate/keypoints_31.csv
32
33
34
35
36
37
38
39
file not found: ./coordinate/keypoints_39.csv
40
41
42
43
44
45
46
file not found: ./coordinate/keypoints_46.csv
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
file not found: ./coordinate/keypoints_71.csv
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
file not found: ./coordinate/keypoints_86.csv
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
file not found: ./coordinate/keypoints_113.csv
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
file not found: ./coordinate/keypoints_150.csv
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188


In [3]:
def standardize_data(data):

    mean = data.mean(axis = (1, 2), keepdims = True)

    std = data.std(axis = (1, 2), keepdims = True)

    standardized_data = (data - mean) / (std + 1e-8)

    return standardized_data

X_standardized = standardize_data(X)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(X_standardized, y, test_size = 0.2, random_state = 13, stratify = y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.2, random_state = 13, stratify = y_temp)

print(f"train shape: {X_train.shape}, validation shape: {X_val.shape}, Test shape: {X_test.shape}")

train shape: (912, 90, 99), validation shape: (182, 90, 99), Test shape: (46, 90, 99)


In [5]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers = 3):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first = True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out
    
input_dim = X_train.shape[2]
hidden_dim = 256
output_dim = len(labels_map)
model = LSTMModel(input_dim = input_dim, hidden_dim = hidden_dim, output_dim = output_dim)

train_dataset = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long())
val_dataset = TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val).long())
train_loader = DataLoader(train_dataset, batch_size = 4, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 4)

In [6]:
from sklearn.utils.class_weight import compute_class_weight

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

class_weights = compute_class_weight("balanced", classes = np.unique(y_train), y=y_train)
criterion = nn.CrossEntropyLoss(weight = torch.tensor(class_weights, dtype = torch.float))

In [7]:
from torch.optim.lr_scheduler import StepLR
import numpy as np

import torch

class EarlyStopping:
    def __init__(self, patience = 7, min_delta = 0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss - val_loss > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0

        else:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True

scheduler = StepLR(optimizer, step_size = 5, gamma = 0.5)

early_stopping = EarlyStopping(patience = 7, min_delta = 0.0001)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        X_batch = X_batch.view(X_batch.size(0), -1, input_dim)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.view(X_batch.size(0), -1, input_dim)
            y_pred = model(X_batch)
            val_loss = criterion(y_pred, y_batch).item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Train_Loss: {train_loss / len(train_loader): .4f}, Validation Loss: {val_loss / len(val_loader): .4f}")

    # early_stopping(val_loss / len(val_loader))

    # if early_stopping.early_stop:
    #     print("Early stopping triggered. Training stopped.")
    #     break

/home/dj/venv/dl_project/lib/python3.12/site-packages/torch/autograd/graph.py:829: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/100, Train_Loss:  0.8323, Validation Loss:  0.0660
Epoch 2/100, Train_Loss:  0.7819, Validation Loss:  0.0481
Epoch 3/100, Train_Loss:  0.6678, Validation Loss:  0.0432
Epoch 4/100, Train_Loss:  0.6770, Validation Loss:  0.0384
Epoch 5/100, Train_Loss:  0.6791, Validation Loss:  0.0434
Epoch 6/100, Train_Loss:  0.5943, Validation Loss:  0.0325
Epoch 7/100, Train_Loss:  0.5766, Validation Loss:  0.0348
Epoch 8/100, Train_Loss:  0.5348, Validation Loss:  0.0706
Epoch 9/100, Train_Loss:  0.5480, Validation Loss:  0.0580
Epoch 10/100, Train_Loss:  0.5155, Validation Loss:  0.0663
Epoch 11/100, Train_Loss:  0.4648, Validation Loss:  0.0116
Epoch 12/100, Train_Loss:  0.4499, Validation Loss:  0.0522
Epoch 13/100, Train_Loss:  0.4161, Validation Loss:  0.0324
Epoch 14/100, Train_Loss:  0.4027, Validation Loss:  0.0623
Epoch 15/100, Train_Loss:  0.4540, Validation Loss:  0.0155
Epoch 16/100, Train_Loss:  0.3785, Validation Loss:  0.0486
Epoch 17/100, Train_Loss:  0.3706, Validation Los

In [8]:
print(f"Data min: {X.min()}, Data Max: {X.max()}")
print(f"Class distribution: np.bincount(y)")

Data min: -1.0058419704437256, Data Max: 1.195205569267273
Class distribution: np.bincount(y)


In [9]:
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []

test_loader = DataLoader(TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).long()), batch_size = 4)

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.view(X_batch.size(0), -1, input_dim)

        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        y_true.extend(y_batch.numpy())
        y_pred.extend(preds.numpy())

print(classification_report(y_true, y_pred, target_names = list(labels_map.keys())))

              precision    recall  f1-score   support

      Normal       0.79      0.73      0.76        15
     Warning       0.78      0.93      0.85        15
        Fall       0.86      0.75      0.80        16

    accuracy                           0.80        46
   macro avg       0.81      0.81      0.80        46
weighted avg       0.81      0.80      0.80        46



In [10]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_true, y_pred))

[[11  2  2]
 [ 1 14  0]
 [ 2  2 12]]


In [11]:
import torch
import torch.nn as nn
import os

scripted_model = torch.jit.script(model)

output_dir = "./"

model_path = os.path.join(output_dir, "lstm_model_scripted.pt")

scripted_model.save(model_path)
print(f"Scripted model saved to {model_path}")

Scripted model saved to ./lstm_model_scripted.pt
